# 02 — NAIP access and epoch inventory

Which NAIP epochs exist where across the pilot tile, and how to pull a chip. This is the group's "look at the imagery" moment and it closes a §17 open item.

**Reads** `cheyenne_corridor_aoi.gpkg` (mgrs_tiles, usgs_gauges)  
**Writes** `naip_epochs_13TFJ.csv`, a chip per example window  
**Status** Phase 2 support — skeleton

> Skeleton. Section headings and the config cell are in place; the analysis cells are deliberately empty for the group to fill in together.

## 0. Setup

In [ ]:
import os, sys

# PROJ/GDAL paths must be set before any geospatial import: the Jupyter kernel starts
# without `conda activate`, so PROJ cannot otherwise find its database.
def _find_share(name):
    for base in (sys.prefix, sys.base_prefix):
        p = os.path.join(base, "share", name)
        if os.path.isdir(p):
            return p
    return None

_proj, _gdal = _find_share("proj"), _find_share("gdal")
if _proj:
    os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = _proj
if _gdal:
    os.environ.setdefault("GDAL_DATA", _gdal)

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import pystac_client, planetary_computer

def _repo_root():
    """Walk up from the working directory to the repo root."""
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / ".git").exists() or (p / "environment.yml").is_file():
            return p
    raise RuntimeError("Could not find the repo root from " + str(here))

REPO     = _repo_root()
DATA_DIR = REPO / "data"
RUNS_DIR = REPO / "runs"          # manifests live here because data/ is gitignored
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Figures ----
# figures/ is gitignored, but tracked via .gitkeep so it exists on a fresh clone —
# nothing to set up after cloning. Contents stay out of git because the repo is
# public and an executed run embeds a 60 cm gallery map (research plan §11).
FIG_DIR = REPO / "figures"
FIG_DPI = 300

def savefig(name, dpi=FIG_DPI):
    """Save the current figure to figures/<FIG_SUBDIR>/<name>.png.

    Call this BEFORE plt.show(): showing a figure can clear it, and you would
    silently save a blank page. FIG_SUBDIR is set in the config cell.
    """
    out = FIG_DIR / FIG_SUBDIR
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{name}.png"
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"figure -> {path.relative_to(REPO)}")
    return path

print("Imports OK")
print(f"  repo : {REPO}")

## 1. Configuration

Every parameter lives here. Pointing this notebook at another tile or another river is a single-cell edit.

In [ ]:
# ---- The pilot tile ----
TILE = "13TFJ"                       # holds Angostura, Buffalo Gap, Red Shirt, Scenic

# ---- Example reaches: one window per 8-digit USGS gauge inside 13TFJ ----
# Gauge-anchored so every window has a flow record to read alongside it (notebook 07).
# These are the walkthrough reaches, NOT the full corridor -- scaling is Phase 6.
EXAMPLE_WINDOWS = [
    {"site_no": "06401500", "name": "Angostura",   "lon": -103.4340, "lat": 43.3470},
    {"site_no": "06402600", "name": "Buffalo Gap", "lon": -103.2350, "lat": 43.4230},
    {"site_no": "06403700", "name": "Red Shirt",   "lon": -102.8921, "lat": 43.6724},
    {"site_no": "06408650", "name": "Scenic",      "lon": -102.5500, "lat": 43.7800},
]
WINDOW_HALF_M = 1000                 # half-width -> 2 x 2 km windows, as in notebook 03
# VERIFY: lon/lat for all but Red Shirt are approximate -- replace from the
# usgs_gauges layer of cheyenne_corridor_aoi.gpkg on first run.

# ---- Catalog ----
# Planetary Computer, signed anonymously -- no account, no key, no egress bill.
# SAS tokens expire in ~45 min: a sudden 403 means re-run the search, not a broken raster.
STAC_URL   = "https://planetarycomputer.microsoft.com/api/stac/v1"
COLLECTION = "naip"
YEARS      = None                    # None -> every epoch the catalog holds (ends 2023)

# ---- Outputs ----
RUN_NAME = f"naip_epochs_{TILE}"
FIG_SUBDIR = RUN_NAME               # figures/<run>/
OUT_DIR  = DATA_DIR / RUN_NAME

## 2. The tile and the example windows

Load the MGRS tile geometry and place the four gauge-anchored windows inside it.

## 3. Search the catalog

One search per window, all years. Collect item ID, date, GSD, and footprint.

## 4. Epoch inventory

A table of year x window: which epochs exist, at what resolution. Red Shirt is known to have 7 (2012/2014 at 1 m; 2016/2018/2020/2021/2022 at 60 cm) — do the others match? Uneven coverage constrains which benchmark epochs notebook 05 can use.

## 5. Read a chip

Windowed read straight out of the COG under `GDAL_DISABLE_READDIR_ON_OPEN=EMPTY_DIR` — nothing downloads a full tile. NAIP is EPSG:26913, VBET is EPSG:32613; take the CRS off the opened raster rather than hardcoding either.

## 6. Save and record the run

Every output gets a manifest in `runs/` — small, text, always committed, even when the raster it describes is not.

In [ ]:
manifest = {
    "run_name":    RUN_NAME,
    "notebook":    "02_NAIP_Access.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "inputs":      {},          # STAC item IDs, upstream run names, source manifests
    "parameters":  {},          # everything from the config cell
    "environment": {"python": sys.version.split()[0]},
    "results":     {},
    "outputs":     [],
}

# manifest_path = RUNS_DIR / f"{RUN_NAME}.manifest.json"
# manifest_path.write_text(json.dumps(manifest, indent=2, default=str) + "\n")

## What comes next

Notebook 03 turns one epoch into training labels. The epoch table here decides which years notebook 05 can use as benchmarks.